In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, cross_val_score

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

from lightgbm import LGBMClassifier

In [2]:
df = pd.read_csv(r"E:\AARAV\Infotact-DS-ML\Project-1-Predictive-Maintenance\data\processed\model_ready_dataset.csv")

df.head()

,HDF,OSF,PWF,TWF,rpm_torque_interaction,Rotational speed [rpm],load_stress,Torque [Nm],load_density,Tool wear [min],temperature_ratio,tool_wear_mean_10,temperature_difference,air_temp_mean_10,UDI,Machine failure
0,0,0,0,0,71177.0,1306,29.7025,54.5,0.545,50,1.034806,36.8,10.4,298.60,19,0
1,0,0,0,0,53040.0,1632,10.5625,32.5,0.325,55,1.034794,40.2,10.4,298.64,20,0
2,0,0,0,0,58712.5,1375,18.2329,42.7,0.427,58,1.034794,43.6,10.4,298.69,21,0
3,0,0,0,0,64960.0,1450,20.0704,44.8,0.448,63,1.035141,47.0,10.5,298.71,22,0
4,0,0,0,0,48536.7,1581,9.4249,30.7,0.307,65,1.034794,50.1,10.4,298.74,23,0


In [3]:
X = df.drop("Machine failure", axis=1)

y = df["Machine failure"]

In [4]:
X.columns = (
    X.columns
    .str.replace("[","",regex=False)
    .str.replace("]","",regex=False)
    .str.replace("{","",regex=False)
    .str.replace("}","",regex=False)
    .str.replace(":","",regex=False)
    .str.replace(",","",regex=False)
)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [5]:
baseline_model = Pipeline(
    steps=[
        ("smote", SMOTE(random_state=42)),
        ("model", LGBMClassifier(random_state=42))
    ]
)

In [6]:
baseline_score = cross_val_score(baseline_model, X, y, cv=skf, scoring="f1")

baseline_score

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 7714, number of negative: 7714
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000705 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2805
[LightGBM] [Info] Number of data points in the train set: 15428, number of used features: 15
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 7714, number of negative: 7714
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000798 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2804
[LightGBM] [Info] Number of data points in the train set: 15428, number of used features: 15
[LightGBM] [Info] [binary:BoostFromScore]: pa

array([0.93617021, 0.89932886, 0.90909091, 0.94285714, 0.87248322])

In [7]:
baseline_score.mean()

np.float64(0.9119860690501843)

In [8]:
params = [
    {
        "n_estimators":100,
        "learning_rate":0.05,
        "num_leaves":31
    },

    {
        "n_estimators":200,
        "learning_rate":0.05,
        "num_leaves":50
    },

    {
        "n_estimators":300,
        "learning_rate":0.03,
        "num_leaves":70
    }
]

In [9]:
tuning_results = []

for p in params:
    model = Pipeline(
        steps=[
            ("smote", SMOTE(random_state=42)),
            ("model", LGBMClassifier(random_state=42,**p))
        ]
    )

    score = cross_val_score(model, X, y, cv=skf, scoring="f1")

    tuning_results.append({
        "parameters": p,
        "mean_f1": score.mean()
    })

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 7714, number of negative: 7714
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000913 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2805
[LightGBM] [Info] Number of data points in the train set: 15428, number of used features: 15
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 7714, number of negative: 7714
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001076 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2804
[LightGBM] [Info] Number of data points in the train set: 15428, number of used features: 15
[LightGBM] [Info] [binary:BoostFromScore]: pa

In [10]:
tuning_df = pd.DataFrame(tuning_results)

tuning_df

,parameters,mean_f1
0,"{'n_estimators': 100, 'learning_rate': 0.05, '...",0.840516
1,"{'n_estimators': 200, 'learning_rate': 0.05, '...",0.926210
2,"{'n_estimators': 300, 'learning_rate': 0.03, '...",0.923367


In [13]:
tuning_df.to_csv(r"E:\AARAV\Infotact-DS-ML\Project-1-Predictive-Maintenance\data\processed\lightgbm_tuning_results.csv", index=False)